[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap08_machine_learning/avaliacao_classificacao_limiar_decisao.ipynb)

# Avaliação da classificação e escolha do limiar de decisão

Este notebook retoma regressão logística e árvore de decisão do exemplo de classificação de pedidos segundo o cumprimento do prazo. A escolha do limiar ocorre somente em um conjunto de validação. O conjunto de teste fica reservado para a avaliação final.

O endereço do botão pressupõe a publicação deste notebook em `capitulos/cap08_machine_learning/avaliacao_classificacao_limiar_decisao.ipynb` no repositório do livro.

## 1. Objetivo e protocolo

Os classificadores produzem probabilidades ou escores para a classe positiva “Fora do prazo”. Um limiar converte esse escore em uma classe prevista. Alterar o limiar pode modificar os falsos positivos e falsos negativos e, consequentemente, a precisão, a sensibilidade e o F1.

A base é dividida em treinamento, validação e teste. Os modelos são ajustados apenas no treinamento. A escolha do limiar é realizada usando validação, sem consultar os resultados do teste. Após a decisão, o teste é usado uma única vez para estimar o desempenho final do procedimento.

In [ ]:
import io
import re
import unicodedata
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    precision_recall_curve, classification_report)

SEMENTE = 2026
URL_BASE = "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
CLASSE_POSITIVA = "Fora do prazo"
np.random.seed(SEMENTE)

## 2. Carregamento da Base Caso Tech

O notebook carrega a planilha do repositório do projeto. Se o download não estiver disponível no ambiente, é possível selecionar o arquivo no Colab. A inspeção das abas e colunas deve anteceder a configuração do alvo.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE, timeout=30) as resposta:
            conteudo = resposta.read()
        return pd.ExcelFile(io.BytesIO(conteudo))
    except Exception as erro:
        print("Download indisponível:", erro)
        try:
            from google.colab import files
            enviados = files.upload()
            if not enviados:
                raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel = carregar_excel()
bases = {aba: pd.read_excel(excel, sheet_name=aba) for aba in excel.sheet_names}
for aba, df in bases.items():
    print(f"{aba}: {df.shape}")
    print(list(df.columns))

## 3. Preparação validada

Para manter consistência com o exemplo anterior, configure a aba, chave de pedido, alvo e preditores usando os nomes exatos observados na planilha. O alvo deve ter duas classes, com a classe positiva definida como “Fora do prazo”. O conjunto de preditores deve conter somente campos disponíveis antes da entrega. Não inclua a variável de alvo, datas de entrega realizada, atraso observado ou outras informações posteriores ao evento.

A unidade deve ser um pedido. Se a planilha tiver várias linhas por pedido, consolide-a conforme a regra validada no exemplo anterior antes desta etapa. Não se deve usar uma regra automática de consolidação sem conferir a granularidade.

In [ ]:
# Preencha após inspecionar as abas e colunas acima.
ABA = None
COLUNA_PEDIDO = None
COLUNA_ALVO = None
PREDITORES_VALIDADOS = None  # lista dos nomes exatos das colunas permitidas
MAPA_ALVO = None  # dicionário de categorias observadas -> 'No prazo' / 'Fora do prazo'

if not all([ABA, COLUNA_PEDIDO, COLUNA_ALVO, PREDITORES_VALIDADOS, MAPA_ALVO]):
    raise ValueError("Preencha ABA, COLUNA_PEDIDO, COLUNA_ALVO, PREDITORES_VALIDADOS e MAPA_ALVO com os valores validados.")

df = bases[ABA].copy()
necessarias = [COLUNA_PEDIDO, COLUNA_ALVO] + PREDITORES_VALIDADOS
faltantes = [c for c in necessarias if c not in df.columns]
if faltantes:
    raise KeyError(f"Colunas não encontradas: {faltantes}")

# Confirme que as classes observadas têm correspondência explícita no mapa.
print("Categorias observadas no alvo:")
display(df[COLUNA_ALVO].value_counts(dropna=False).to_frame("frequência"))
y_texto = df[COLUNA_ALVO].astype(str).map(MAPA_ALVO)
if y_texto.isna().any():
    categorias_sem_mapa = df.loc[y_texto.isna(), COLUNA_ALVO].astype(str).unique().tolist()
    raise ValueError(f"Categorias sem mapeamento validado: {categorias_sem_mapa}")

base = df[[COLUNA_PEDIDO] + PREDITORES_VALIDADOS].copy()
base["alvo"] = y_texto
base = base.dropna(subset=[COLUNA_PEDIDO, "alvo"])
if base[COLUNA_PEDIDO].duplicated().any():
    raise ValueError("Há mais de uma linha por pedido. Consolide por pedido com regra justificada antes de prosseguir.")
if set(base["alvo"].unique()) != {"No prazo", CLASSE_POSITIVA}:
    raise ValueError("O alvo mapeado deve conter exatamente 'No prazo' e 'Fora do prazo'.")

X = base[PREDITORES_VALIDADOS].copy()
y = base["alvo"].copy()
print("Pedidos:", len(base))
display(y.value_counts().to_frame("frequência"))

## 4. Separação de treinamento, validação e teste

A separação é estratificada e reprodutível. O teste é separado antes do ajuste de modelos e da escolha do limiar. Treinamento serve para estimar os parâmetros dos classificadores; validação serve para comparar limiares; teste serve apenas para a avaliação final. Em dados temporais, ou quando pedidos compartilham cliente, rota ou lote, substitua a divisão aleatória por uma estratégia que respeite essa dependência.

In [ ]:
X_dev, X_teste, y_dev, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=SEMENTE, stratify=y_dev
)
print("Treinamento:", len(X_treino), "Validação:", len(X_validacao), "Teste:", len(X_teste))
print("Proporção aproximada: 60% / 20% / 20%")

## 5. Pré-processamento e ajuste dos modelos

O pré-processamento é ajustado dentro de cada pipeline, usando somente dados de treinamento. Variáveis numéricas recebem imputação pela mediana e padronização; variáveis categóricas recebem imputação pela categoria mais frequente e codificação one-hot. A regressão logística e a árvore de decisão usadas aqui retomam os modelos do exemplo anterior, com parâmetros didáticos definidos previamente.

In [ ]:
num = X_treino.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X_treino.columns if c not in num]
preprocessador = ColumnTransformer([
    ("num", Pipeline([("imputacao", SimpleImputer(strategy="median")),
                      ("escala", StandardScaler())]), num),
    ("cat", Pipeline([("imputacao", SimpleImputer(strategy="most_frequent")),
                      ("codificacao", OneHotEncoder(handle_unknown="ignore"))]), cat)
])

modelos = {
    "Regressão logística": LogisticRegression(max_iter=2000),
    "Árvore de decisão": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, random_state=SEMENTE
    )
}
pipelines = {}
for nome, estimador in modelos.items():
    pipe = Pipeline([("preprocessamento", preprocessador), ("modelo", estimador)])
    pipe.fit(X_treino, y_treino)
    pipelines[nome] = pipe
print("Modelos ajustados no treinamento.")

## 6. Escolha do limiar na validação

Para cada modelo, obtém-se no conjunto de validação o escore da classe positiva. Quando o estimador oferece probabilidades, usa-se a probabilidade estimada. A regra inicial de seleção escolhe o limiar que maximiza o F1 da classe positiva na validação. Em caso de empate, o código seleciona o primeiro limiar na ordem avaliada. Essa regra é uma decisão didática; em uma aplicação, o critério deve refletir o objetivo e os custos operacionais conhecidos.

O teste não participa dessa escolha. Não se ajusta o limiar depois de examinar o resultado no teste.

In [ ]:
def obter_escore(pipe, X_parte, classe_positiva):
    modelo = pipe.named_steps["modelo"]
    if hasattr(pipe, "predict_proba"):
        classes = list(pipe.classes_)
        if classe_positiva not in classes:
            raise ValueError("Classe positiva não encontrada no modelo.")
        return pipe.predict_proba(X_parte)[:, classes.index(classe_positiva)]
    raise TypeError("O modelo não fornece probabilidades para seleção de limiar.")

def selecionar_limiar_f1(y_real, escores, classe_positiva):
    limiares = np.unique(np.r_[0.0, escores, 1.0])
    resultados = []
    for limiar in limiares:
        pred = np.where(escores >= limiar, classe_positiva, "No prazo")
        resultados.append(f1_score(y_real, pred, pos_label=classe_positiva, zero_division=0))
    indice = int(np.argmax(resultados))
    return float(limiares[indice]), float(resultados[indice]), limiares, resultados

limiares_escolhidos = {}
escores_validacao = {}
for nome, pipe in pipelines.items():
    escore = obter_escore(pipe, X_validacao, CLASSE_POSITIVA)
    limiar, f1_valid, limiares, f1s = selecionar_limiar_f1(
        y_validacao.to_numpy(), escore, CLASSE_POSITIVA
    )
    limiares_escolhidos[nome] = limiar
    escores_validacao[nome] = (limiares, f1s)
    print(f"{nome}: limiar={limiar:.3f}; F1 de validação={f1_valid:.3f}")

## 7. Curva F1 por limiar e matriz de confusão na validação

A curva permite visualizar como o F1 varia conforme o limiar. A matriz de confusão na validação é apresentada para o limiar selecionado, mas não substitui a avaliação final no teste. Precisão, sensibilidade (revocação) e F1 são reportados para a classe positiva.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nome, (limiares, f1s) in escores_validacao.items():
    ax.plot(limiares, f1s, label=nome)
    ax.axvline(limiares_escolhidos[nome], linestyle="--", alpha=0.5)
ax.set_xlabel("Limiar de decisão")
ax.set_ylabel("F1 na validação")
ax.set_title("Escolha do limiar usando somente validação")
ax.legend()
plt.tight_layout()
plt.show()

for nome, pipe in pipelines.items():
    escore = obter_escore(pipe, X_validacao, CLASSE_POSITIVA)
    pred = np.where(escore >= limiares_escolhidos[nome], CLASSE_POSITIVA, "No prazo")
    cm = confusion_matrix(y_validacao, pred, labels=["No prazo", CLASSE_POSITIVA])
    print("\nValidação:", nome, "| limiar:", round(limiares_escolhidos[nome], 3))
    print("Precisão:", round(precision_score(y_validacao, pred, pos_label=CLASSE_POSITIVA, zero_division=0), 3))
    print("Sensibilidade:", round(recall_score(y_validacao, pred, pos_label=CLASSE_POSITIVA, zero_division=0), 3))
    print("F1:", round(f1_score(y_validacao, pred, pos_label=CLASSE_POSITIVA, zero_division=0), 3))
    ConfusionMatrixDisplay(cm, display_labels=["No prazo", CLASSE_POSITIVA]).plot(values_format="d")
    plt.title(f"Validação: {nome}")
    plt.show()

## 8. Avaliação final no teste

Após registrar os limiares escolhidos na validação, aplica-se cada limiar aos escores do teste. O conjunto de teste não é usado para escolher o modelo, alterar o limiar ou modificar o pré-processamento. Os resultados abaixo constituem a avaliação final do procedimento definido.

In [ ]:
resultados_teste = []
for nome, pipe in pipelines.items():
    escore_teste = obter_escore(pipe, X_teste, CLASSE_POSITIVA)
    limiar = limiares_escolhidos[nome]
    pred = np.where(escore_teste >= limiar, CLASSE_POSITIVA, "No prazo")
    resultados_teste.append({
        "Modelo": nome,
        "Limiar escolhido na validação": limiar,
        "Precisão (teste)": precision_score(y_teste, pred, pos_label=CLASSE_POSITIVA, zero_division=0),
        "Sensibilidade (teste)": recall_score(y_teste, pred, pos_label=CLASSE_POSITIVA, zero_division=0),
        "F1 (teste)": f1_score(y_teste, pred, pos_label=CLASSE_POSITIVA, zero_division=0),
        "Acurácia (teste)": accuracy_score(y_teste, pred)
    })
    cm = confusion_matrix(y_teste, pred, labels=["No prazo", CLASSE_POSITIVA])
    print("\nTeste:", nome, "| limiar fixado:", round(limiar, 3))
    ConfusionMatrixDisplay(cm, display_labels=["No prazo", CLASSE_POSITIVA]).plot(values_format="d")
    plt.title(f"Teste: {nome}")
    plt.show()

display(pd.DataFrame(resultados_teste).set_index("Modelo").round(3))

## 9. Custos hipotéticos: extensão opcional

Não há custos operacionais fornecidos para a Base Caso Tech. Portanto, não se atribuem valores monetários reais. A célula seguinte é uma análise de sensibilidade hipotética: os custos são parâmetros ilustrativos, não estimativas empíricas nem recomendações. Altere-os somente para explorar como diferentes penalidades relativas afetam o limiar na validação. O teste continua reservado e não participa da escolha.

In [ ]:
# Exemplo hipotético, sem correspondência com custos reais da operação.
CUSTO_FALSO_NEGATIVO = 5.0  # atraso previsto como “No prazo”
CUSTO_FALSO_POSITIVO = 1.0  # pedido no prazo previsto como “Fora do prazo”
print("Custos hipotéticos declarados:")
print("Falso negativo:", CUSTO_FALSO_NEGATIVO)
print("Falso positivo:", CUSTO_FALSO_POSITIVO)

# Busca um limiar de menor custo médio na validação para cada modelo.
# Não usar esses parâmetros como custos reais sem dados e validação com responsáveis.
for nome, pipe in pipelines.items():
    escore = obter_escore(pipe, X_validacao, CLASSE_POSITIVA)
    candidatos = np.unique(np.r_[0.0, escore, 1.0])
    custos = []
    for t in candidatos:
        pred = np.where(escore >= t, CLASSE_POSITIVA, "No prazo")
        cm = confusion_matrix(y_validacao, pred, labels=["No prazo", CLASSE_POSITIVA])
        tn, fp, fn, tp = cm.ravel()
        custo_medio = (CUSTO_FALSO_POSITIVO * fp + CUSTO_FALSO_NEGATIVO * fn) / len(y_validacao)
        custos.append(custo_medio)
    i = int(np.argmin(custos))
    print(f"{nome}: limiar hipotético de menor custo na validação={candidatos[i]:.3f}; custo médio hipotético={custos[i]:.3f}")

## 10. Interpretação e checklist

O limiar padrão de 0,5 não é necessariamente o mais adequado. Aumentar o limiar tende a reduzir previsões positivas, o que pode elevar a precisão e reduzir a sensibilidade; o efeito observado deve ser avaliado nos dados. A escolha baseada em F1 dá peso combinado à precisão e à sensibilidade, mas não representa automaticamente os custos do processo.

### Checklist

- O alvo e a classe positiva foram mapeados a partir das categorias reais?
- Há exatamente um registro por pedido?
- Todos os preditores estariam disponíveis no momento previsto para classificação?
- Treinamento, validação e teste foram separados antes de selecionar o limiar?
- O limiar foi escolhido somente na validação?
- O teste foi usado apenas para a avaliação final?
- Custos, caso usados, estão identificados como hipotéticos e não como valores reais?